# core

> Scheme implementation in Python

`eval`, `apply`, and `env` are the central loop of a Lisp interpreter:

- **`env`** answers: “what does this symbol mean?”
- **`eval`** answers: “what is the value of this expression in this environment?”
- **`apply`** answers: “given a function/procedure and evaluated arguments, how do I call it?”

For a simple expression:

```scheme
(+ x 3)
```

the flow is:

1. `eval` sees a list, so it treats it as a procedure call.
2. It evaluates the first element, `+`, by looking it up in `env`.
3. It evaluates each argument: `x` is looked up in `env`, `3` evaluates to itself.
4. Then `eval` hands the resulting procedure and values to `apply`.
5. `apply` actually calls the procedure.

Conceptually:

```python
proc = eval('+', env)
args = [eval('x', env), eval(3, env)]
return apply(proc, args)
```

So `eval` walks and interprets expressions; `apply` performs calls; `env` gives meaning to symbols along the way.

In [ ]:
#| default_exp core

In [ ]:
import math, operator as op
import inspect
from fastcore.basics import store_attr, first, last

from compact.reader import *
from compact.types import *

### Environment

In Lisp/Scheme, an **env** is the “memory of names”: it maps symbols like `x`, `+`, or `square` to the values they currently mean.

For example, when evaluating:

```scheme
(+ x 3)
```

the evaluator sees `+` and `x` as symbols. It needs an environment to answer:

- what function does `+` refer to?
- what value does `x` refer to?

So conceptually:

```python
env = {'x': 10, '+': operator.add}
```

Then `(+ x 3)` can become “call add on 10 and 3”.

The important idea is: **expressions don’t carry all their meaning alone; symbols get their meaning from the environment they’re evaluated in.**

Later, `env` also lets us support local bindings:

```scheme
(let ((x 5))
  (+ x 1))
```

Inside the `let`, `x` means `5`; outside, it might mean something else. That means environments often form a **chain**: look in the local env first, then the outer env if not found.

_In our case since the goal is to produce an embedded lisp that interops with python, we are going to share globals() (the python symbols) with env used by the lisp evaluator._

In [ ]:
class Env:
    "implementation of env for scheme eval()."
    def __init__(self, d, primitives:dict=(), parent=None): store_attr(cast=True)

    def __getitem__(self, k):
        s = self.find(k)
        if s is not None: 
            # we currently do not allow shadowing of primitives - this is not ideal
            if k in s.primitives: return s.primitives[k]
            return s.d[k]
        raise KeyError(k)

    def __setitem__(self, k, v): self.d[k] = v

    def __contains__(self, k): return self.find(k) is not None

    def update(self, d): self.primitives.update(d)

    def find(self, k):
        if k in self.d | self.primitives: return self
        if self.parent is not None: return self.parent.find(k)
        return None
    
    def is_primitive(self, k):
        if self.parent is None: return k in self.primitives
        return self.parent.is_primitive(k)

    def new_frame(self, bindings=()):
        return Env(dict(bindings), parent=self)

### The Evaluator: eval and apply
The two-function core: `eval` dispatches, `apply` calls.

In [ ]:
def scm_apply(fn, args):
    kw = {}
    if len(args) > 0 and isinstance(args[-1], KwArgs): 
        kw.update(args[-1])
        args = args[:-1]
    if isinstance(fn, Procedure): return fn._thunk(*args)
    return fn(*args, **kw)

In [ ]:
class Procedure:
    "a dual use wrapper for scm_eval as well as python callers"
    def __init__(self, params, body, env, sfs): store_attr()
    def _thunk(self, *args): 
        "return a Thunk so this can be executed inside the eval loop"
        return Thunk(_body_expr(self.body), self.env.new_frame(_bind_params(self.params, args)))
    def __call__(self, *args): 
        "if called directly from python, eval it for the caller"
        return scm_eval_tco(_body_expr(self.body), 
            self.env.new_frame(_bind_params(self.params, args)), self.sfs)

In [ ]:
def scm_eval_one_step(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, list): return expr

    # lists
    hd, body = expr[0], expr[1:]

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)
    fn = scm_eval_tco(hd, env, sfs)
    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)

    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])

In [ ]:
def scm_eval_tco(expr, env, sfs=()):
    "eval but with tail call optimization"
    while True:
        r = scm_eval_one_step(expr, env, sfs)
        if isinstance(r, Thunk): expr, env = r.expr, r.env
        else: return r

Lisp code is mostly recursive. 

Naive recursive evaluation stack overflows for deep recursion (e.g. `(fact 10000)`). Instead of recursing, we use something called Tail Call Optimization which can be used to convert recursion into iteration. Calls return `Thunk`s that can be executed in the same stack frame. This is also called trampolining.

In our case, the main eval loop is split into two parts that call each other and eventually apply.


### Primitive Procedures

Primitive procedures are built in and cannot be redefined — attempting `(define + ...)` raises a `SyntaxError`.

**Arithmetic**

| Operator | Meaning |
|---|---|
| `+` `-` `*` `/` | Variadic; `-` negates if unary, `/` inverts if unary |
| `=` `<` `>` `<=` `>=` | Numeric comparison |
| `abs` `min` `max` `expt` `sqrt` | Common numeric operations |
| `floor` `ceiling` `round` `truncate` | Rounding |
| `modulo` `remainder` | Integer division remainder (differ on negatives) |

**Lists**

| Operator | Meaning |
|---|---|
| `list` `cons` `car` `cdr` | Construction and access |
| `null?` `pair?` `list?` | Predicates |

**Strings**

| Operator | Meaning |
|---|---|
| `string-append` `string-length` `substring` | Operations |
| `number->string` `string->number` | Numeric conversion |
| `symbol->string` `string->symbol` | Symbol conversion |

**Predicates & equality**

| Operator | Meaning |
|---|---|
| `number?` `string?` `symbol?` `boolean?` | Type predicates |
| `equal?` | Deep structural equality (type-strict) |
| `not` | Boolean negation |

**Other**

| Operator | Meaning |
|---|---|
| `error` | Raise an exception with a message and optional irritants |

In [ ]:
def _scm_error(msg, *irritants): raise Exception(msg if not irritants else f"{msg} {' '.join(map(repr, irritants))}")

In [ ]:
def _scm_sub(x, *xs): return x - sum(xs) if xs else -x
def _scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

In [ ]:
def _is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def _is_sym(x): return isinstance(x, Symbol)
def _is_sym_eq(x, s): return _is_sym(x) and x.s == s
def _is_list(x): return isinstance(x, list)
def _is_pair(x): return isinstance(x, list) and bool(x)

In [ ]:
def _scm_equal(x, y):
    if type(x) != type(y): return False
    if isinstance(x, list): return len(x) == len(y) and all(_scm_equal(a,b) for a,b in zip(x,y))
    return x == y

In [ ]:
def _str2num(s):
    try: return int(s)
    except ValueError:
        try: return float(s)
        except ValueError: return False

In [ ]:
def _scm_for_each(fn, *lsts):
    for args in zip(*lsts): fn(*args)

In [ ]:
def _scm_fold_left(fn, init, lst):
    acc = init
    for x in lst: acc = fn(acc, x)
    return acc

In [ ]:
def _scm_fold_right(fn, init, lst):
    acc = init
    for x in reversed(lst): acc = fn(x, acc)
    return acc

In [ ]:
_builtin = {
    "+": lambda *xs: sum(xs), 
    "*": lambda *xs: math.prod(xs),
    "-": _scm_sub, "/": _scm_div,
    "=": lambda x,y: _is_num(x) and _is_num(y) and x == y,
    "<": op.lt, ">": op.gt, "<=": op.le, ">=": op.ge,
}
_builtin |= {
    "number?": _is_num, "string?": lambda x: isinstance(x, str),
    "symbol?": _is_sym, "boolean?": lambda x: isinstance(x, bool),
    "null?": lambda x: x == [], "pair?": _is_pair, "list?": _is_list,
}
_builtin |= {
    "list": lambda *xs: list(xs), "cons": lambda x,y: [x] + y,
    "car": lambda x: x[0], "cdr": lambda x: x[1:],
}
_builtin |= {
    "string-append": lambda *xs: "".join(xs),
    "string-length": len,
    "substring": lambda s,i,j: s[i:j],
    "number->string": str, "string->number": _str2num,
}
_builtin |= {
    "floor": math.floor, "ceiling": math.ceil,
    "round": round, "truncate": math.trunc,
    "abs": abs, "min": min, "max": max,
    "sqrt": math.sqrt, "expt": pow, "modulo": op.mod,
    "remainder": lambda x,y: x - int(x/y)*y,
}
_builtin |= {"symbol->string": lambda x: x.s, "string->symbol": Symbol,
    "equal?": _scm_equal,
    "not": lambda x: x is False, "error": _scm_error}
_builtin |= {
    "apply": lambda fn, *args: fn(*args[:-1], *args[-1]),
    "map": lambda fn, *lsts: [fn(*args) for args in zip(*lsts)],
    "filter": lambda fn, lst: [x for x in lst if fn(x) is not False],
    "for-each": _scm_for_each,
    "fold-left": _scm_fold_left,
    "fold-right": _scm_fold_right,
}

We need a way to access the evaluator from inside python. `LispCtx` bundles the evaluator, the special forms registry, and the primitive procedures into a single object. 

The `@` operator (`"expr" @ lisp`) is the primary interface — it evaluates a Lisp expression in an environment that includes the calling module's globals, so Python variables are visible to Lisp and anything `define`d lands back in Python.

`lisp[name]` looks up a symbol in the same environment — useful for symbols with names that are not valid Python identifiers (e.g. `lisp['string->number']`).


In [ ]:
class LispCtx:
    "convenience lisp context for interop"
    def __init__(self):
        self.env = Env(globals(), primitives=_builtin)
        self.sfs = {}

    def sf(self, nm=None):
        def reg(fn):
            key = nm or fn.__name__.removeprefix('_sf_').replace('_', '-')
            self.sfs[key] = fn; return fn
        return reg

    def __rmatmul__(self, s):
        caller_globals = inspect.stack()[1][0].f_globals
        env = Env(caller_globals, primitives=self.env.primitives, parent=self.env)
        exprs = parse_all(s)
        expr = exprs[0] if len(exprs) == 1 else [Symbol("begin")] + exprs
        return scm_eval_tco(expr, env, self.sfs)

    def __getitem__(self, k):
        caller_globals = inspect.stack()[1][0].f_globals
        env = Env(caller_globals, primitives=self.env.primitives, parent=self.env)
        return env[k]

    def register_magic(self):        
        from IPython.core.magic import register_cell_magic
        caller_globals = inspect.stack()[1][0].f_globals
        def _magic(line, cell):
            exprs = parse_all(cell)
            expr = exprs[0] if len(exprs) == 1 else [Symbol("begin")] + exprs
            env = Env(caller_globals, primitives=self.env.primitives, parent=self.env)
            return scm_eval_tco(expr, env, self.sfs)
        register_cell_magic('lisp')(_magic)

`lisp` is the module-level singleton. `lisp` can be imported and directly used. The singleton captures `core.py`'s globals at import time; each `@` call layers the caller's globals on top via a child frame.

In [ ]:
lisp = LispCtx()

### Special Forms

Special forms look like procedure calls but do not evaluate all arguments eagerly —
each form decides when and how to evaluate its sub-expressions.

**Core**

| Form | Syntax | Meaning |
|---|---|---|
| `quote` | `'x` | Return expression unevaluated |
| `define` | `(define name val)` / `(define (f a…) body)` | Bind a symbol in the current environment |
| `lambda` | `(lambda (a…) body)` | Anonymous procedure |
| `begin` | `(begin e …)` | Evaluate sequence, return last |
| `set!` | `(set! name val)` | Mutate an existing binding |

**Control flow**

| Form | Syntax | Meaning |
|---|---|---|
| `if` | `(if test then else)` | Conditional; only the taken branch is evaluated |
| `cond` | `(cond (test expr) … (else expr))` | Multi-branch conditional |
| `when` | `(when test body…)` | Evaluate body only if test is truthy |
| `unless` | `(unless test body…)` | Evaluate body only if test is false |
| `case` | `(case val ((v…) expr)… (else expr))` | Dispatch on a value against literal lists |
| `and` | `(and e …)` | Short-circuit; returns last value or `#f` |
| `or` | `(or e …)` | Short-circuit; returns first truthy value or `#f` |

**Binding**

| Form | Syntax | Meaning |
|---|---|---|
| `let` | `(let ((x v) …) body)` | Local bindings (all evaluated in outer env) |
| `let` (named) | `(let loop ((i 0)) body)` | Named let — local recursive loop |
| `let*` | `(let* ((x v) …) body)` | Sequential bindings; each sees the previous |
| `letrec` | `(letrec ((x v) …) body)` | Recursive bindings; each can reference the others |

**Lists**

| Form | Syntax | Meaning |
|---|---|---|
| `apply` | `(apply fn arg… lst)` | Call `fn` spreading `lst` as its arguments |
| `map` | `(map fn lst…)` | Apply `fn` across one or more lists |
| `filter` | `(filter fn lst)` | Keep elements where `fn` returns truthy |
| `for-each` | `(for-each fn lst…)` | Like `map` but for side effects |
| `fold-left` | `(fold-left fn init lst)` | Reduce left-to-right with accumulator |
| `fold-right` | `(fold-right fn init lst)` | Reduce right-to-left with accumulator |

**Macros**

| Form | Syntax | Meaning |
|---|---|---|
| `macro` | `(macro (a…) body)` | Define a code transformer |
| `quasiquote` | `` `(… ,x ,@xs) `` | Template with `,` splice and `,@` list splice |

**Python interop**

| Form | Syntax | Meaning |
|---|---|---|
| `->` | `(-> obj attr)` | Access a Python attribute or bound method |
| `@` | `(@ (k v) …)` | Build a kwargs map; trailing arg to any Python callable |

In [ ]:
@lisp.sf()
def _sf_quote(xs, env, sfs): return xs[0]

In [ ]:
@lisp.sf()
def _sf_begin(xs, env, sfs):
    for o in xs[:-1]: scm_eval_tco(o, env, sfs)
    return Thunk(xs[-1], env)

In [ ]:
def _body_expr(body): return body[0] if len(body) == 1 else [Symbol("begin")] + body

def _bind_params(ps, vs):
    match ps:
        case Symbol(s=nm): return {nm: list(vs)}
        case [*fixed, Symbol(s="."), Symbol(s=rest)]:
            return dict(zip([p.s for p in fixed], vs)) | {rest: list(vs[len(fixed):])}
        case _:
            if len(vs) != len(ps): raise ValueError(f"arity mismatch: wanted {len(ps)}, got {len(vs)}")
            return dict(zip([p.s for p in ps], vs))

In [ ]:
@lisp.sf()
def _sf_if(xs, env, sfs):
    cond,then_,else_ = xs
    br = else_ if scm_eval_tco(cond, env, sfs) is False else then_
    return Thunk(br, env)

In [ ]:
@lisp.sf()
def _sf_lambda(xs, env, sfs): return Procedure(xs[0], xs[1:], env, sfs)

In [ ]:
@lisp.sf()
def _sf_define(xs, env, sfs):
    def _mkfn(sym, expr):
        if not isinstance(sym, Symbol): raise SyntaxError(f"{sym} must be a symbol")
        if env.is_primitive(sym.s): raise SyntaxError(f"cannot redefine primitive '{sym.s}'")
        env[sym.s] = scm_eval_tco(expr, env, sfs)
        return sym.s
    arg0, *rest = xs
    if isinstance(arg0, Symbol): return _mkfn(arg0, _body_expr(rest))
    if isinstance(arg0, list): return _mkfn(arg0[0], [Symbol("lambda"), arg0[1:]] + rest)

In [ ]:
@lisp.sf("set!")
def _sf_set(xs, env, sfs):
    sym, expr = xs
    if not isinstance(sym, Symbol): raise SyntaxError(f"set! argument {sym} must be a symbol")
    e = env.find(sym.s)
    if e is None: raise NameError(sym.s)

    e[sym.s] = scm_eval_tco(expr, env, sfs)
    return sym.s

In [ ]:
@lisp.sf()
def _sf_let(xs, env, sfs):
    ev = lambda x: scm_eval_tco(x, env, sfs)
    match xs:
        case [Symbol(s=nm), binds, *body]:  # (let loop ((i 0)) body)
            pairs = [(o[0], ev(o[1])) for o in binds]
            params, inits = zip(*pairs) if pairs else ([], [])
            new_env = env.new_frame({nm: None})
            new_env[nm] = _sf_lambda([list(params)] + body, new_env, sfs)
            return new_env[nm](*inits)
        case [binds, *body]:                # (let ((x 1)) body)
            new_env = env.new_frame({n.s: ev(v) for n,v in binds})
            return Thunk(_body_expr(body), new_env)

In [ ]:
@lisp.sf()
def _sf_cond(xs, env, sfs):
    for test, *body in xs:
        if _is_sym_eq(test, 'else') or scm_eval_tco(test, env, sfs) is not False:
            return Thunk(_body_expr(body), env)

In [ ]:
@lisp.sf()
def _sf_and(xs, env, sfs):
    if not xs: return True
    for o in xs[:-1]:
        if scm_eval_tco(o, env, sfs) is False: return False
    return Thunk(xs[-1], env)

@lisp.sf()
def _sf_or(xs, env, sfs):
    if not xs: return False
    for o in xs[:-1]:
        v = scm_eval_tco(o, env, sfs)
        if v is not False: return v
    return Thunk(xs[-1], env)

In [ ]:
@lisp.sf("let*")
def _sf_let_star(xs, env, sfs):
    binds, *body = xs
    for name, val in binds:
        env = env.new_frame({name.s: scm_eval_tco(val, env, sfs)})
    return Thunk(_body_expr(body), env)

In [ ]:
@lisp.sf()
def _sf_letrec(xs, env, sfs):
    binds, *body = xs
    new_env = env.new_frame({name.s: None for name, _ in binds})
    for name, val in binds: new_env[name.s] = scm_eval_tco(val, new_env, sfs)
    return Thunk(_body_expr(body), new_env)

In [ ]:
@lisp.sf()
def _sf_case(xs, env, sfs):
    key = scm_eval_tco(xs[0], env, sfs)
    for vals, *body in xs[1:]:
        if _is_sym_eq(vals, 'else') or key in vals: return Thunk(_body_expr(body), env)

In [ ]:
@lisp.sf("macro")
def _macro(args, env, sfs):
    params, *body = args
    return Macro(lambda *vals: scm_eval_tco(_body_expr(body), env.new_frame(_bind_params(params, vals)), sfs))

In [ ]:
def _qq(x, env, sfs, depth=1):
    "handle ` , and ,@"
    QQ, UQ, UQS = Symbol("quasiquote"), Symbol("unquote"), Symbol("unquote-splicing")
    def tagged(o, sym): return isinstance(o, list) and len(o) == 2 and o[0] == sym

    if not isinstance(x, list): return x
    if tagged(x, QQ): return [QQ, _qq(x[1], env, sfs, depth+1)]
    if tagged(x, UQ):
        return scm_eval_tco(x[1], env, sfs) if depth == 1 else [UQ, _qq(x[1], env, sfs, depth-1)]
    if tagged(x, UQS):
        if depth == 1: raise SyntaxError("unquote-splicing only valid inside a list")
        return [UQS, _qq(x[1], env, sfs, depth-1)]

    def walk(lst):
        result = []
        for o in lst:
            if tagged(o, UQS) and depth == 1: result.extend(scm_eval_tco(o[1], env, sfs))
            else: result.append(_qq(o, env, sfs, depth))
        return result
    return walk(x)

In [ ]:
"""
(define when (macro (cond . body) `(if ,cond (begin ,@body) #f)))
(define unless (macro (cond . body) `(if ,cond #f (begin ,@body))))
""" @ lisp

'unless'

In [ ]:
@lisp.sf()
def _sf_quasiquote(args, env, sfs): return _qq(args[0], env, sfs)

In [ ]:
@lisp.sf("@")
def _sf_at(args, env, sfs):
    return KwArgs({k.s: scm_eval_tco(v, env, sfs) for k,v in args})

In [ ]:
@lisp.sf("->")
def _sf_dot(args, env, sfs):
    obj_expr, m = args
    obj = scm_eval_tco(obj_expr, env, sfs)
    return getattr(obj, m.s)

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()